# Softmax

Softmax takes a vector of raw scores (logits) and turns them into **probabilities that sum to 1**.

Where sigmoid handles binary (one value → probability), softmax handles **multi-class** (many values → probability distribution).

## 1. Softmax Function

**Formula:**

```
softmax(x_i) = e^(x_i) / Σ e^(x_j)
```

For each element: raise e to that value, then divide by the sum of all e^values.

**What it does:**
- Input: any vector of real numbers (can be negative, large, small)
- Output: same-length vector of values between 0 and 1 that sum to 1

**When to use:**
- Multi-class classification (which category?)
- Attention scores (which tokens to focus on?)
- LLM next-token prediction (which word comes next?)
- RAG retrieval ranking (which document is most relevant?)

In [ ]:
import math

def softmax(x):
    exp_values = [math.exp(val) for val in x]
    total = sum(exp_values)
    return [val / total for val in exp_values]

scores = [2.0, 1.0, 0.1]
probs = softmax(scores)

print("Softmax Function")
print("-" * 40)
print(f"Input scores: {scores}")
print(f"Probabilities: {[round(p, 4) for p in probs]}")
print(f"Sum:           {sum(probs):.4f}")

## 2. Step by Step

Let's walk through softmax([2.0, 1.0, 0.1]) by hand.

In [ ]:
scores = [2.0, 1.0, 0.1]

print("Step 1: Compute e^x for each score")
print("-" * 45)
exp_values = []
for i, s in enumerate(scores):
    e = math.exp(s)
    exp_values.append(e)
    print(f"  e^{s} = {e:.4f}")

print(f"\nStep 2: Sum all e^x values")
print("-" * 45)
total = sum(exp_values)
print(f"  {' + '.join(f'{e:.4f}' for e in exp_values)} = {total:.4f}")

print(f"\nStep 3: Divide each by the sum")
print("-" * 45)
for i, (s, e) in enumerate(zip(scores, exp_values)):
    prob = e / total
    print(f"  softmax({s}) = {e:.4f} / {total:.4f} = {prob:.4f}  ({prob:.1%})")

print(f"\nResult: the highest score (2.0) gets the most probability")
print(f"But the others still get some — softmax is 'soft', not winner-take-all")

## 3. Visualizing Different Distributions

How softmax shapes different inputs into probability distributions.

In [ ]:
def show_distribution(name, scores, labels=None):
    probs = softmax(scores)
    if labels is None:
        labels = [f"class {i}" for i in range(len(scores))]
    
    print(f"\n{name}")
    print(f"  Scores: {scores}")
    print(f"  {'Label':<12} {'Score':>6} {'Prob':>6}  Bar")
    print(f"  {'-'*50}")
    for label, score, prob in zip(labels, scores, probs):
        bar = '█' * int(prob * 40)
        print(f"  {label:<12} {score:>6.1f} {prob:>6.1%}  {bar}")

# Equal scores → uniform distribution
show_distribution(
    "Equal scores (uniform)",
    [1.0, 1.0, 1.0],
    ["cat", "dog", "bird"]
)

# One dominant score
show_distribution(
    "One dominant score (confident)",
    [5.0, 1.0, 0.5],
    ["cat", "dog", "bird"]
)

# Close scores → uncertain
show_distribution(
    "Close scores (uncertain)",
    [2.0, 1.8, 1.9],
    ["cat", "dog", "bird"]
)

# Negative scores work too
show_distribution(
    "Negative scores (still works)",
    [-1.0, -2.0, 0.5],
    ["cat", "dog", "bird"]
)

## 4. Softmax vs Sigmoid

| | Sigmoid | Softmax |
|---|---|---|
| **Input** | Single number | Vector of numbers |
| **Output** | Single probability | Vector of probabilities |
| **Sum** | N/A | Always sums to 1 |
| **Use case** | Binary: yes/no | Multi-class: which one? |
| **Competition** | No — each output is independent | Yes — increasing one decreases others |

In [ ]:
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

scores = [2.0, 1.0, 0.1]

print("Sigmoid vs Softmax on the same scores")
print(f"Input: {scores}")
print("-" * 55)
print(f"{'Score':>6} | {'Sigmoid':>10} | {'Softmax':>10} | Note")
print("-" * 55)

sig_results = [sigmoid(s) for s in scores]
soft_results = softmax(scores)

for s, sig, soft in zip(scores, sig_results, soft_results):
    print(f"{s:>6.1f} | {sig:>10.4f} | {soft:>10.4f} |")

print(f"{'Sum':>6} | {sum(sig_results):>10.4f} | {sum(soft_results):>10.4f} |")
print(f"\nSigmoid: each value is independent (sum can be anything)")
print(f"Softmax: values compete (sum is always 1.0)")
print(f"\nUse sigmoid when: 'Is this spam?' (yes/no)")
print(f"Use softmax when: 'Is this cat, dog, or bird?' (pick one)")

## 5. Temperature

Temperature controls how **sharp** or **flat** the distribution is.
This is exactly the temperature slider in ChatGPT / Claude.

```
softmax(x_i / T)
```

- **Low temperature (T < 1)** → sharper, more confident, less random
- **T = 1** → standard softmax
- **High temperature (T > 1)** → flatter, more uniform, more random

In [ ]:
def softmax_with_temp(x, temperature=1.0):
    scaled = [val / temperature for val in x]
    exp_values = [math.exp(val) for val in scaled]
    total = sum(exp_values)
    return [val / total for val in exp_values]

scores = [3.0, 1.5, 0.5]
labels = ["hello", "world", "foo"]
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]

print("Temperature Effect on Next-Token Prediction")
print(f"Raw scores: {scores}  (tokens: {labels})")
print("=" * 65)

for temp in temperatures:
    probs = softmax_with_temp(scores, temp)
    bars = ['█' * int(p * 30) for p in probs]
    
    if temp < 0.5:
        desc = "almost deterministic"
    elif temp < 1.0:
        desc = "focused"
    elif temp == 1.0:
        desc = "standard"
    elif temp < 3.0:
        desc = "creative"
    else:
        desc = "very random"
    
    print(f"\n  T={temp:<4} ({desc})")
    for label, prob, bar in zip(labels, probs, bars):
        print(f"    {label:<8} {prob:>6.1%}  {bar}")

In [ ]:
# What temperature actually does to the math

scores = [3.0, 1.5, 0.5]

print("Why temperature works: dividing scores before softmax")
print("=" * 55)

for temp in [0.5, 1.0, 3.0]:
    scaled = [s / temp for s in scores]
    probs = softmax(scaled)
    print(f"\n  T={temp}")
    print(f"  Original scores: {scores}")
    print(f"  Divided by T:    [{', '.join(f'{s:.1f}' for s in scaled)}]")
    print(f"  After softmax:   [{', '.join(f'{p:.3f}' for p in probs)}]")

print("\nLow T → scores spread apart → softmax becomes sharp")
print("High T → scores squeeze together → softmax becomes flat")

## 6. Where Softmax Appears

Softmax shows up in three key places relevant to RAG and LLMs.

### 6a. Attention — which tokens to focus on

In [ ]:
# In attention: softmax turns raw dot-product scores into weights

tokens = ["The", "cat", "sat", "on", "the", "mat"]
# Pretend these are dot-product scores: how much "sat" attends to each token
raw_scores = [0.5, 2.8, 1.0, 0.3, 0.4, 0.2]

attention_weights = softmax(raw_scores)

print('Attention: how much does "sat" focus on each word?')
print("-" * 50)
print(f"{'Token':<8} {'Raw score':>10} {'Attention':>10}  Focus")
print("-" * 50)
for token, score, weight in zip(tokens, raw_scores, attention_weights):
    bar = '█' * int(weight * 30)
    print(f"{token:<8} {score:>10.1f} {weight:>10.1%}  {bar}")

print(f"\n'sat' pays most attention to 'cat' — makes sense!")
print(f"Softmax ensures all attention weights sum to 1.")

### 6b. LLM output — picking the next token

In [ ]:
# The LLM outputs raw scores (logits) for every possible next token
# Softmax converts them to probabilities

next_token_candidates = ["runs", "sleeps", "eats", "the", "quickly"]
logits = [4.2, 3.8, 2.1, 0.5, 1.3]  # raw model output

probs = softmax(logits)

print('Prompt: "The cat ___"')
print("LLM picks the next token using softmax on logits")
print("-" * 55)
print(f"{'Token':<10} {'Logit':>6} {'Probability':>12}  Distribution")
print("-" * 55)

# Sort by probability
ranked = sorted(zip(next_token_candidates, logits, probs), key=lambda x: -x[2])
for token, logit, prob in ranked:
    bar = '█' * int(prob * 40)
    print(f"{token:<10} {logit:>6.1f} {prob:>12.1%}  {bar}")

print(f"\nThe model samples from this distribution.")
print(f"Temperature adjusts how peaked or flat this distribution is.")

### 6c. RAG retrieval — ranking documents

In [ ]:
# In RAG: similarity scores between query and documents
# Softmax turns them into a relevance distribution

query = "How to train a neural network?"
documents = [
    "Backpropagation tutorial",
    "Cooking recipes",
    "Gradient descent explained",
    "Weather forecast",
    "Learning rate scheduling",
]
# Pretend these are cosine similarity scores from embeddings
similarity_scores = [0.85, 0.12, 0.78, 0.05, 0.72]

relevance = softmax([s * 10 for s in similarity_scores])  # scale up for sharper ranking

print(f'Query: "{query}"')
print("-" * 65)
print(f"{'Document':<30} {'Similarity':>10} {'Relevance':>10}")
print("-" * 65)

ranked = sorted(zip(documents, similarity_scores, relevance), key=lambda x: -x[2])
for doc, sim, rel in ranked:
    bar = '█' * int(rel * 30)
    print(f"{doc:<30} {sim:>10.2f} {rel:>10.1%}  {bar}")

print(f"\nSoftmax highlights the top matches and suppresses the irrelevant ones.")
print(f"The model can then focus its attention on the top-ranked documents.")

## 7. Numerical Stability

A practical problem: `e^x` explodes for large x values.

In [ ]:
# The problem: large inputs cause overflow

print("The overflow problem")
print("-" * 40)
print(f"e^10   = {math.exp(10):.0f}")
print(f"e^100  = {math.exp(100):.2e}")
print(f"e^500  = {math.exp(500):.2e}")

try:
    result = math.exp(1000)
    print(f"e^1000 = {result}")
except OverflowError:
    print(f"e^1000 = OVERFLOW ERROR!")

print(f"\nIn real models, logits can be large numbers.")
print(f"Naive softmax would crash.")

In [ ]:
# The fix: subtract the max value before computing exp
# This is mathematically equivalent but numerically stable

def softmax_stable(x):
    max_val = max(x)
    shifted = [val - max_val for val in x]  # now the largest value is 0
    exp_values = [math.exp(val) for val in shifted]
    total = sum(exp_values)
    return [val / total for val in exp_values]

# Proof: same result, no overflow
scores_normal = [2.0, 1.0, 0.1]
scores_large = [1000.0, 999.0, 998.1]  # would overflow without the trick

print("The fix: subtract max before exp")
print("=" * 50)

print(f"\nSmall scores: {scores_normal}")
print(f"  Naive:  {[round(p, 4) for p in softmax(scores_normal)]}")
print(f"  Stable: {[round(p, 4) for p in softmax_stable(scores_normal)]}")
print(f"  Same result!")

print(f"\nLarge scores: {scores_large}")
try:
    print(f"  Naive:  {softmax(scores_large)}")
except OverflowError:
    print(f"  Naive:  OVERFLOW!")
print(f"  Stable: {[round(p, 4) for p in softmax_stable(scores_large)]}")
print(f"  Stable version handles it!")

print(f"\nWhy it works:")
print(f"  softmax([1000, 999, 998.1])")
print(f"  = softmax([1000-1000, 999-1000, 998.1-1000])")
print(f"  = softmax([0, -1, -1.9])")
print(f"  Subtracting a constant doesn't change the ratios.")

## 8. NumPy Version (Vectorized)

In [ ]:
import numpy as np

def softmax_np(x, temperature=1.0):
    x = np.array(x) / temperature
    x = x - np.max(x)  # numerical stability
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)

scores = np.array([2.0, 1.0, 0.1])

print("NumPy vectorized softmax")
print(f"Input:      {scores}")
print(f"Softmax:    {np.round(softmax_np(scores), 4)}")
print(f"T=0.5:      {np.round(softmax_np(scores, 0.5), 4)}")
print(f"T=2.0:      {np.round(softmax_np(scores, 2.0), 4)}")
print(f"Sum:        {softmax_np(scores).sum():.4f}")

## Summary

| | Softmax |
|---|---|
| **Formula** | `e^(x_i) / Σ e^(x_j)` |
| **Input** | Vector of any real numbers |
| **Output** | Probabilities summing to 1 |
| **vs Sigmoid** | Multi-class (competition) vs binary (independent) |
| **Temperature** | Low = sharp/confident, High = flat/random |
| **Stability trick** | Subtract max before exp |

### Where softmax appears

| Context | Role |
|---|---|
| **Attention** | Converts Q·K dot products into attention weights |
| **LLM output** | Converts logits into next-token probabilities |
| **RAG ranking** | Converts similarity scores into relevance distribution |
| **Classification** | Converts final layer into class probabilities |